In [25]:
import numpy as np

In [40]:
x = 'ATGCCG'
y = 'ATCG'

def smith_waterman(X, Y, match, mismatch, gap):
    X = list(X)
    Y = list(Y)
    matrix = np.zeros((len(X) + 2, len(Y) + 2), dtype=object)

    for i in range(len(X)):
        matrix[i + 2, 0] = X[i]
    for j in range(len(Y)):
        matrix[0, j + 2] = Y[j]

 
    max_score = 0
    max_pos = (0, 0)

    for i in range(len(X)):
        for j in range(len(Y)):
            up    = matrix[i + 1, j + 2] + gap  
            left  = matrix[i + 2, j + 1] + gap   
            diag  = matrix[i + 1, j + 1] + (match if matrix[i + 2, 0] == matrix[0, j + 2] else mismatch)
            score = max(0, up, left, diag)
            matrix[i + 2, j + 2] = score
            if score > max_score:
                max_score = score
                max_pos = (i + 2, j + 2)

    return matrix, max_score, max_pos


In [41]:
waterman_smith(x, y, match=3, mismatch= -3, gap=-2)


array([[0., 0., 0., 0., 0.],
       [0., 3., 1., 0., 0.],
       [0., 1., 6., 4., 2.],
       [0., 0., 4., 3., 7.],
       [0., 0., 2., 7., 5.],
       [0., 0., 0., 5., 4.],
       [0., 0., 0., 3., 8.]])

In [42]:
def waterman_smith(X, Y, match, mismatch, gap):
    X = list(X)
    Y = list(Y)
    matrix = np.zeros((len(X) + 1, len(Y) + 1))
    
    for i in range(1, len(X) + 1):
        for j in range(1, len(Y) + 1):
            if X[i-1] == Y[j-1]:
                matrix[i, j] = max(0, matrix[i-1, j-1] + match, matrix[i-1, j] + gap, matrix[i, j-1] + gap)
            else:
                matrix[i, j] = max(0, matrix[i-1, j-1] + mismatch, matrix[i-1, j] + gap, matrix[i, j-1] + gap)
    
    return matrix

def traceback(X, Y, matrix, match, mismatch, gap):
    X = list(X)
    Y = list(Y)
    
    i, j = np.unravel_index(np.argmax(matrix), matrix.shape)
    max_score = matrix[i, j]
    
    align1 = []
    align2 = []
    
    while i > 0 and j > 0 and matrix[i, j] > 0:
        current_score = matrix[i, j]
        
        if X[i-1] == Y[j-1]:
            score = match
        else:
            score = mismatch
        
        if current_score == matrix[i-1, j-1] + score:
            align1.append(X[i-1])
            align2.append(Y[j-1])
            i -= 1
            j -= 1
        elif current_score == matrix[i-1, j] + gap:
            align1.append(X[i-1])
            align2.append('-')
            i -= 1
        elif current_score == matrix[i, j-1] + gap:
            align1.append('-')
            align2.append(Y[j-1])
            j -= 1
        else:
            break
    
    align1 = ''.join(reversed(align1))
    align2 = ''.join(reversed(align2))
    
    return align1, align2, max_score

x = 'ATGCCG'
y = 'ATCG'
matrix = waterman_smith(x, y, 2, 0, -1)
print("Matrix:")
print(matrix)
print("\nTraceback:")
a1, a2, score = traceback(x, y, matrix, 2, 0, -1)
print(a1)
print(a2)
print(f"Score: {score}")

Matrix:
[[0. 0. 0. 0. 0.]
 [0. 2. 1. 0. 0.]
 [0. 1. 4. 3. 2.]
 [0. 0. 3. 4. 5.]
 [0. 0. 2. 5. 4.]
 [0. 0. 1. 4. 5.]
 [0. 0. 0. 3. 6.]]

Traceback:
ATGCCG
AT--CG
Score: 6.0


In [39]:
#признаюсь честно: матрицы строил сам, трейсбеки вайбкодил